# 🗺️ Station Query → Map

Ask a natural-language question about seismic stations (e.g. *"Stations within 5 km of Anchorage, Alaska"*), send it to an LLM via [`fdsn-agent`](../fdsn_agent) (built on top of `query.py`), and plot the stations it finds on a `folium` map.

**How it works:** `fdsn-agent` translates your question into an FDSN station-service query, runs it against IRIS, and hands back structured JSON (`result.data["stations"]`). This notebook turns that JSON into a `pandas` DataFrame and drops a marker on the map for each station.

**Requirements:**
- `fdsn_agent` installed (editable install from the sibling `fdsn_agent/` package: `pip install -e ../fdsn_agent`)
- An LLM backend reachable from `LLMConfig` — this notebook defaults to a local Ollama server (`ollama serve`, with the model pulled, e.g. `ollama pull gemma4`). Swap `LLMConfig.from_preset(...)` for `"anthropic"`, `"openai"`, etc. to use a different provider.

**How to play:** Edit `QUERY` in the second code cell to whatever station question you want, then run all cells.

In [1]:
# --- Environment check: run this first ---
def _check():
    missing = []
    for pkg in ["fdsn_agent", "folium", "pandas"]:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)

    if missing:
        print("⚠️  Missing packages:", ", ".join(missing))
        if "fdsn_agent" in missing:
            print("    Run: pip install -e ../fdsn_agent   (editable install of the local package)")
        other = [p for p in missing if p != "fdsn_agent"]
        if other:
            print(f"    Run: pip install {' '.join(other)}")
    else:
        import fdsn_agent
        print("✅ fdsn_agent, folium, pandas are all installed.")
        print(f"fdsn_agent: {fdsn_agent.__version__}")

_check()

✅ fdsn_agent, folium, pandas are all installed.
fdsn_agent: 0.1.0


In [2]:
from fdsn_agent import Agent, LLMConfig

# Local Ollama by default -- swap the preset/model to use a different provider
# (see LLMConfig.from_preset in fdsn_agent_DOCS.md for "anthropic", "openai", etc.)
cfg = LLMConfig.from_preset("ollama", model="gemma4")
agent = Agent(cfg)

QUERY = "Stations within 5 km of Anchorage, Alaska"

In [3]:
import folium
import pandas as pd


def get_station_dataframe(result):
    """Turn an fdsn_station AgentResult into a deduplicated station DataFrame.

    Raises ValueError if the LLM routed the query to a different tool (e.g. it
    read the question as an earthquake or waveform request instead of a
    station search), or if the station search came back empty.

    Deduplicates on (network, station, latitude, longitude): the FDSN station
    service returns one row per deployment epoch, so a station with several
    instrument swaps over its lifetime otherwise produces multiple overlapping
    markers at the same coordinates.
    """
    if result.tool_called != "fdsn_station":
        raise ValueError(
            f"Expected the fdsn_station tool, but the LLM called {result.tool_called!r} "
            "instead. Try rephrasing the query to make it clearer you're asking about "
            "station metadata rather than earthquakes or waveforms."
        )

    stations = result.data.get("stations", [])
    if not stations:
        raise ValueError("No stations returned for this query -- try widening it.")

    df = pd.DataFrame(stations)
    df = df.drop_duplicates(subset=["Network", "Station", "Latitude", "Longitude"])
    df = df.rename(columns=str.lower).reset_index(drop=True)
    for col in ("latitude", "longitude", "elevation"):
        df[col] = df[col].astype(float)
    return df


def build_station_map(df):
    """Build a folium map with a marker per station, auto-fit to their extent."""
    center = [df["latitude"].mean(), df["longitude"].mean()]
    m = folium.Map(location=center, tiles="OpenStreetMap")

    for _, row in df.iterrows():
        popup_html = (
            f"<b>{row['network']}.{row['station']}</b><br>"
            f"{row.get('sitename', '')}<br>"
            f"Elevation: {row['elevation']:.0f} m<br>"
            f"{row.get('starttime', '?')} – {row.get('endtime', '?')}"
        )
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=5,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"{row['network']}.{row['station']}",
            color="#1f77b4",
            fill=True,
            fill_opacity=0.7,
        ).add_to(m)

    bounds = [
        [df["latitude"].min(), df["longitude"].min()],
        [df["latitude"].max(), df["longitude"].max()],
    ]
    m.fit_bounds(bounds)
    return m


print("Functions loaded.")

Functions loaded.


In [4]:
result = agent.query(QUERY)

print(f"Query: {QUERY}")
print(f"Tool called: {result.tool_called}({result.tool_params})")
print(f"Summary: {result.summary}\n")

station_df = get_station_dataframe(result)
print(f"{len(station_df)} unique station(s) found.")
station_df.head()

Query: Stations within 5 km of Anchorage, Alaska
Tool called: fdsn_station({'latitude': 61.2181, 'longitude': -149.9003, 'maxradius': 0.05})
Summary: {
  "summary": "The search successfully identified 35 seismological stations near Anchorage, Alaska. These stations belong to multiple networks (AK, NP, GD, SY) and cover various sites within a 5 km radius of the specified coordinates.",
  "data": {
    "count": 35,
    "stations": [
      {
        "Network": "AK",
        "Station": "K201",
        "Latitude": "61.234402",
        "Longitude": "-149.871201",
        "Elevation": "36.0",
        "SiteName": "Anchorage, High Lat Monitoring Station, AK, USA",
        "StartTime": "2009-07-01T00:00:00.0000",
        "EndTime": "2009-10-31T23:59:59.0000"
      },
      {
        "Network": "AK",
        "Station": "K202",
        "Latitude": "61.223801",
        "Longitude": "-149.809006",
        "Elevation": "47.0",
        "SiteName": "Anchorage, Park and Recreation",
        "StartTime":

,network,station,latitude,longitude,elevation,sitename,starttime,endtime
0,AK,K201,61.234402,-149.871201,36.0,"Anchorage, High Lat Monitoring Station, AK, USA",2009-07-01T00:00:00.0000,2009-10-31T23:59:59.0000
1,AK,K202,61.223801,-149.809006,47.0,"Anchorage, Park and Recreation",2009-07-01T00:00:00.0000,2010-12-31T23:59:59.0000
2,AK,K205,61.199600,-149.913696,27.0,"Anchorage, ASD Data Processing Center, AK, USA",1995-08-01T00:00:00.0000,2013-01-23T23:59:59.0000
3,AK,K205,61.199402,-149.916306,51.0,"Anchorage, West High School, AK, USA",2013-01-24T00:00:00.0000,
4,AK,K208,61.176300,-149.921600,20.0,"Anchorage, Spenard Recreation Center, AK, USA",1995-08-01T00:00:00.0000,


In [5]:
build_station_map(station_df)